# California Residential Home Price Prediction
# Preprocessing Pipeline

This notebook prepares the cleaned, model-ready dataset from the raw CRMLS MLS data explored in `01_exploration.ipynb`.

Decisions made here are documented with reasoning — each choice is intentional and defensible based on findings from the EDA phase.

## Section 1 - Setup & Data Loading

Raw CRMLS MLS data is loaded from local CSV files and filtered to Single Family 
Residential properties only, as specified in the task requirements.

In [24]:
import pandas as pd
import numpy as np
import glob
import os

files = glob.glob('/Users/granthbangard/Local Files/Documents/granth/UIUC/IDXExchange/DataSet/CRMLSSold*.csv')
df = pd.concat([pd.read_csv(f, low_memory = False) for f in files], ignore_index = True)
print('Full dataset shape:', df.shape)

Full dataset shape: (794271, 82)


In [25]:
df_sfr = df[(df['PropertyType'] == 'Residential') & (df['PropertySubType'] == 'SingleFamilyResidence')].copy()
print(df_sfr.shape)

(399157, 82)


## Section 2 - Filtering & Date Cutoff

Based on EDA findings, records before June 2023 have very low transaction counts (under 100/month) 
consistent with incomplete data ingestion rather than actual market activity. These records are 
excluded to ensure the model trains on reliable, representative data.

In [26]:
df_sfr['CloseDate'] = pd.to_datetime(df_sfr['CloseDate'], errors='coerce')
df_sfr = df_sfr[df_sfr['CloseDate'] >= '2023-06-01'].copy()
print('Shape after date cutoff:', df_sfr.shape)

Shape after date cutoff: (391673, 82)


## Section 3 - Dropping Useless Columns

Columns are dropped for one of three reasons:
- Completely empty after SFR filter (no usable signal)
- Data leakage (contain information only available after the sale)
- Not relevant to property price prediction (agent/office identifiers, administrative fields)

In [27]:
cols_to_drop = [
    # Leakage
    'ListPrice', 'OriginalListPrice', 'DaysOnMarket', 'PurchaseContractDate',
    # Completely empty
    'ElementarySchoolDistrict', 'MiddleOrJuniorSchoolDistrict',
    'WaterfrontYN', 'BasementYN', 'latfilled', 'lonfilled',
    # Agent/office identifiers
    'ListAgentEmail', 'ListAgentFirstName', 'ListAgentLastName',
    'ListAgentFullName', 'ListOfficeName', 'CoListOfficeName',
    'CoListAgentFirstName', 'CoListAgentLastName',
    'BuyerAgentFirstName', 'BuyerAgentLastName', 'BuyerAgentMlsId',
    'BuyerOfficeName', 'BuyerOfficeAOR', 'CoBuyerAgentFirstName',
    'BuyerAgencyCompensation', 'BuyerAgencyCompensationType',
    'ListAgentAOR', 'BuyerAgentAOR',
    # Administrative/irrelevant
    'ListingKey', 'ListingKeyNumeric', 'ListingId', 'BusinessType',
    'TaxYear', 'BuilderName', 'LotSizeDimensions', 'AboveGradeFinishedArea',
    'BelowGradeFinishedArea', 'TaxAnnualAmount', 'BuildingAreaTotal',
    'SubdivisionName', 'MlsStatus', 'StateOrProvince', 'PropertyType',
    'PropertySubType', 'ContractStatusChangeDate', 'ElementarySchool',
    'HighSchool', 'AssociationFeeFrequency', 'MainLevelBedrooms',
    'StreetNumberNumeric', 'LotSizeAcres', 'LotSizeArea', 'Flooring',
    'CoveredSpaces', 'ParkingTotal', 'FireplaceYN', 'MiddleOrJuniorSchool'
]

df_sfr = df_sfr[df_sfr['StateOrProvince'] == 'CA'].copy()
df_sfr = df_sfr.drop(columns=cols_to_drop)
print('Shape after dropping columns:', df_sfr.shape)
print('Remaining columns:', df_sfr.columns.tolist())

Shape after dropping columns: (391649, 25)
Remaining columns: ['ViewYN', 'PoolPrivateYN', 'CloseDate', 'ClosePrice', 'Latitude', 'Longitude', 'UnparsedAddress', 'LivingArea', 'FireplacesTotal', 'MLSAreaMajor', 'CountyOrParish', 'AttachedGarageYN', 'YearBuilt', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'ListingContractDate', 'Stories', 'Levels', 'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee', 'LotSizeSquareFeet']


## Section 4 - Outlier Removal

Two outlier thresholds applied based on EDA findings:

- **Lower bound:** Records below $10,000 removed — 10 records with prices this low 
  are almost certainly data errors or partial interest transfers, not real home sales.
- **Upper bound:** Records above $5,000,000 removed — prices above this threshold 
  include clear data anomalies (records up to $989M exist in the raw data) that would 
  distort model training. California's legitimate luxury SFR market is well-represented 
  below this threshold.

In [28]:
print("Shape before removal:", df_sfr.shape)
df_sfr = df_sfr[(df_sfr['ClosePrice'] >= 10000) & (df_sfr['ClosePrice'] <= 5000000)].copy()
print('Shape after outlier removal:', df_sfr.shape)

Shape before removal: (391649, 25)
Shape after outlier removal: (385572, 25)


## Section 5 - Missing Value Treatment

Missing values are handled on a column-by-column basis depending on the nature of the field:

- **Latitude/Longitude:** 34 records missing — recovered via geocoding using UnparsedAddress 
  rather than imputation, as median coordinates would place properties at incorrect locations
- **Numeric features:** Imputed with median — robust to remaining outliers
- **Categorical/Boolean features:** Imputed with mode
- **AssociationFee:** Missing likely means no HOA — imputed with 0

In [29]:
df_sfr = df_sfr.drop(columns=['FireplacesTotal'])

df_sfr = df_sfr.dropna(subset=['PostalCode', 'UnparsedAddress'])

df_sfr['AssociationFee'] = df_sfr['AssociationFee'].fillna(0)
df_sfr['NewConstructionYN'] = df_sfr['NewConstructionYN'].fillna('No')

for col in ['LivingArea', 'YearBuilt', 'BathroomsTotalInteger', 
            'LotSizeSquareFeet', 'GarageSpaces', 'Stories']:
    df_sfr[col] = df_sfr[col].fillna(df_sfr[col].median())

for col in ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'Levels', 'MLSAreaMajor', 'City']:
    df_sfr[col] = df_sfr[col].fillna(df_sfr[col].mode()[0])

print('Shape:', df_sfr.shape)
print('Remaining nulls:')
print(df_sfr.isnull().sum())

Shape: (385202, 24)
Remaining nulls:
ViewYN                       0
PoolPrivateYN                0
CloseDate                    0
ClosePrice                   0
Latitude                    34
Longitude                   34
UnparsedAddress              0
LivingArea                   0
MLSAreaMajor                 0
CountyOrParish               0
AttachedGarageYN             0
YearBuilt                    0
BathroomsTotalInteger        0
City                         0
BedroomsTotal                0
ListingContractDate          0
Stories                      0
Levels                       0
NewConstructionYN            0
GarageSpaces                 0
HighSchoolDistrict       98838
PostalCode                   0
AssociationFee               0
LotSizeSquareFeet            0
dtype: int64


/var/folders/n_/hd4whr8972b54sv4t6_db9d80000gn/T/ipykernel_10877/1114102914.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sfr[col] = df_sfr[col].fillna(df_sfr[col].mode()[0])


In [30]:
# Dropped due to high cardinality (1088 unique values) and most frequent value being "Not Defined"
df_sfr = df_sfr.drop(columns=['MLSAreaMajor'])

In [31]:
# Geocoding 34 records with missing coordinates using UnparsedAddress — recovered 15, remaining 19 dropped

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="idx_home_price")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

missing_coords = df_sfr[df_sfr['Latitude'].isnull()].copy()

for idx, row in missing_coords.iterrows():
    try:
        location = geocode(row['UnparsedAddress'])
        if location:
            df_sfr.at[idx, 'Latitude'] = location.latitude
            df_sfr.at[idx, 'Longitude'] = location.longitude
    except Exception as e:
        print(f"Failed for index {idx}: {e}")

print('Remaining missing coords:', df_sfr['Latitude'].isnull().sum())

Remaining missing coords: 19


In [42]:
df_sfr = df_sfr.dropna(subset=['Latitude', 'Longitude'])
print('Shape:', df_sfr.shape)

Shape: (384961, 25)


## Section 6 - Feature Engineering

New features derived from existing columns to improve model signal:

- **PropertyAge:** Years since construction at time of sale — more interpretable than raw YearBuilt
- **YearBuilt:** Dropped after PropertyAge is derived — redundant
- **BedBathRatio:** Bedrooms divided by bathrooms — captures home layout efficiency
- **PricePerSqFt:** Not used as a feature (would cause leakage since it's derived from ClosePrice) — noted here to avoid confusion
- **CloseMonth / CloseYear:** Extracted from CloseDate to capture seasonality and year-over-year trends
- **ListingDuration:** Days between ListingContractDate and CloseDate — measures how long the property was on market before closing

In [33]:
df_sfr["PropertyAge"] = df_sfr["CloseDate"].dt.year - df_sfr["YearBuilt"]
df_sfr['BedBathRatio'] = df_sfr['BedroomsTotal'] / df_sfr['BathroomsTotalInteger'].replace(0, np.nan)
df_sfr["CloseMonth"] = df_sfr["CloseDate"].dt.month
df_sfr["CloseYear"] = df_sfr["CloseDate"].dt.year
df_sfr['ListingContractDate'] = pd.to_datetime(df_sfr['ListingContractDate'], errors='coerce')
df_sfr["ListingDuration"] = (df_sfr["CloseDate"] - df_sfr["ListingContractDate"]).dt.days

df_sfr[['PropertyAge', 'BedBathRatio', 'CloseMonth', 'CloseYear', 'ListingDuration']].describe()

,PropertyAge,BedBathRatio,CloseMonth,CloseYear,ListingDuration
count,385183.000000,385036.000000,385183.000000,385183.000000,385183.000000
mean,49.227902,1.472147,6.623893,2024.410537,71.840844
std,27.248733,0.475300,3.298169,0.931664,66.704026
min,-1.000000,0.000000,1.000000,2023.000000,-31.000000
25%,28.000000,1.000000,4.000000,2024.000000,36.000000
50%,49.000000,1.500000,7.000000,2024.000000,53.000000
75%,69.000000,1.666667,9.000000,2025.000000,86.000000
max,249.000000,7.500000,12.000000,2026.000000,14758.000000


In [34]:
df_sfr = df_sfr[(df_sfr['ListingDuration'] >= 0) & (df_sfr['ListingDuration'] <= 730)].copy()
print('Shape:', df_sfr.shape)

Shape: (384961, 28)


In [ ]:
df_sfr = df_sfr.drop(columns=['ListingContractDate', 'CloseDate', 'YearBuilt'])
print('Shape:', df_sfr.shape)

Shape: (384961, 25)


## Section 7 - Encoding Categorical Variables

Categorical and boolean columns are converted to numeric representations for model compatibility:

- **Boolean columns** (ViewYN, PoolPrivateYN, AttachedGarageYN): already True/False, no action needed — XGBoost handles these natively
- **Levels:** Simplified from messy multi-value strings to a clean ordinal (1 = one story, 2 = two story, 3 = three or more)
- **NewConstructionYN:** Dropped — 90% missing after Yes/No mapping, no usable signal
- **CountyOrParish, City, PostalCode, HighSchoolDistrict:** Label encoded — high cardinality columns converted to integer codes

A copy of the preprocessed dataframe is saved as `df_model` before encoding to preserve the option of trying alternative encoding strategies without rerunning the full pipeline.

In [36]:
def simplify_levels(val):
    if pd.isna(val): return 1
    if 'Three' in str(val) or 'Multi' in str(val): return 3
    if 'Two' in str(val): return 2
    return 1

df_sfr['Levels'] = df_sfr['Levels'].apply(simplify_levels)

df_sfr['NewConstructionYN'] = df_sfr['NewConstructionYN'].map({'Yes': 1, 'No': 0})

In [37]:
from sklearn.preprocessing import LabelEncoder

df_model = df_sfr.copy()
df_model = df_model.drop(columns=['UnparsedAddress'])

le = LabelEncoder()
for col in ['CountyOrParish', 'City', 'PostalCode', 'HighSchoolDistrict']:
    df_model[col] = df_model[col].fillna('Unknown')
    df_model[col] = le.fit_transform(df_model[col])

print('Shape:', df_model.shape)

Shape: (384961, 24)


In [38]:
df_model = df_model.drop(columns=['NewConstructionYN'])
print('Shape:', df_model.shape)

Shape: (384961, 23)


## Section 8 - Train/Test Split

A temporal split is used rather than a random split to prevent data leakage. 
Training on future data to predict past prices would produce artificially inflated 
metrics that don't reflect real-world performance.

- **Test set:** Most recent full month of available data
- **Training set:** All preceding months (training window length treated as a tunable parameter)

The most recent month in the dataset is identified programmatically so this pipeline 
remains valid as new monthly data is appended.

In [39]:
latest_month = df_model['CloseYear'] * 100 + df_model['CloseMonth']
test_month = latest_month.max()
print(f'Test month: {test_month}')

test_mask = latest_month == test_month
df_train = df_model[~test_mask].copy()
df_test = df_model[test_mask].copy()

X_train = df_train.drop(columns=['ClosePrice'])
y_train = df_train['ClosePrice']
X_test = df_test.drop(columns=['ClosePrice'])
y_test = df_test['ClosePrice']

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

Test month: 202605
Training set: (373162, 22)
Test set: (11799, 22)


In [40]:
# Save cleaned, encoded dataset for use in modeling notebooks
# Note: this file is gitignored and must be regenerated by running this notebook

df_model.to_csv('data/cleaned_data.csv', index=False)
print('Saved.')

Saved.
